# 03 — Train/test và CNN FedAdam theo số mẫu train

Cell cuối chỉ chạy CNN FedAdam với trọng số theo số mẫu fit thực tế. Giữ toàn bộ cấu hình CNN đã chọn, bao gồm client Adam LR 0,0003. Thử 20 server learning rate, mỗi giá trị 15 vòng × 3 local epoch; chọn F1 Attack validation trung bình ba client ở vòng cuối. Khởi tạo lại từ đầu với LR tốt nhất rồi đánh giá test một lần. Kết quả và tuning lưu riêng dưới `fedadam_weighted`.

In [4]:
from pathlib import Path
import os, sys, json, importlib
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p/'COPY_PROVENANCE.json').is_file()
               and (p/'configs/tuning.json').is_file() and (p/'notebook_workflow.py').is_file())
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for module_name in ('common', 'feature_preprocessing', 'prepare', 'models', 'metrics', 'notebook_workflow'):
    module = importlib.import_module(module_name)
    assert Path(module.__file__).resolve().parent == PROJECT.resolve(), 'Restart kernel to load this copy.'
    importlib.reload(module)
workflow = sys.modules['notebook_workflow']
CONFIG = PROJECT/'configs/tuning.json'
C = workflow.settings(CONFIG)
RUN_ROOT = workflow.run_root(C)
print('Project:', PROJECT)
print('Results:', RUN_ROOT)
display(workflow.budget(CONFIG))

Project: .
Results: runs/cic_all_attack_dirichlet_a05_max1p5m


{'tasks': ['binary'],
 'tuning_task': 'binary',
 'models': ['dnn', 'cnn'],
 'trials_per_model': 10,
 'rounds': 15,
 'local_epochs': 3,
 'baseline_epochs': 45,
 'client_optimizer': 'adam',
 'server_optimizer': 'sgd',
 'server_learning_rate': 1.0,
 'tuning_weighting': 'number_of_examples',
 'final_weightings': ['number_of_examples', 'uniform'],
 'selection': 'mean_cross_client_validation_f1',
 'checkpoint_selection': 'final_round',
 'total_tuning_trials': 20,
 'fits_per_trial': '3 Local + 1 Centralized + 1 sample-weighted FedAvg',
 'objective': 'mean of 12 validation F1: 6 cross-client Local + 3 Centralized + 3 FL',
 'trial_1': {'dnn': {'architecture': '8-4-2',
   'dropout': 0.5,
   'batch_size': 1024,
   'client_learning_rate': 0.0005},
  'cnn': {'architecture': '8-16-8',
   'kernel_size': 3,
   'dropout': 0.2,
   'batch_size': 2048,
   'client_learning_rate': 0.0005}}}

In [2]:
SCENARIOS = ["local_cic_1", "local_cic_2", "local_cic_3", "centralized", "fedavg_weighted"]

In [3]:
for model in C["models"]:
    display(workflow.selected_params(C,model,task="binary"))
    frame = workflow.run_experiments(model,CONFIG,scenarios=SCENARIOS,task="binary")
    display(frame[frame["scenario"] != "fedavg_uniform"])

{'study_identity': '4f71a14595de6e98fd10d2cfbb32b2ee9d6110fd0c88106f592111b41f32f981',
 'task': 'binary',
 'model': 'dnn',
 'trial': 14,
 'params': {'architecture': '16-8-4',
  'dropout': 0.3,
  'batch_size': 4096,
  'client_learning_rate': 0.0005},
 'rank': [0.7622448240725608, 0.6908694297288875, 0.7734320253593564],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'tuning_scenario': 'fedavg_weighted',
 'tuning_scenarios': ['local_cic_1',
  'local_cic_2',
  'local_cic_3',
  'centralized',
  'fedavg_weighted'],
 'objective': 'mean_cross_client_validation_f1'}

2026-11-03 19:34:00.909213: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 19:34:00.909260: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-03 19:34:00.909279: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-03 19:34:03.155749: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 19:34:03.155807: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register f

,task,model,scenario,client_weighting,test_client,samples,accuracy,macro_precision,macro_recall,macro_f1,...,far,precision,recall,f1,tn,fp,fn,tp,auc,pr_auc
0,binary,dnn,centralized,None,cic_1,146314,0.878829,0.923630,0.759865,0.803888,...,0.002723,0.984562,0.522453,0.682657,109516,299,17430,19069,0.788332,0.729077
1,binary,dnn,centralized,None,cic_2,276449,0.994165,0.994919,0.992548,0.993712,...,0.001333,0.997683,0.986429,0.992024,174529,233,1380,100307,0.993901,0.994455
2,binary,dnn,centralized,None,cic_3,26396,0.936998,0.590688,0.561772,0.572599,...,0.851363,0.959701,0.974906,0.967244,180,1031,632,24553,0.916769,0.995879
3,binary,dnn,centralized,None,pooled,449159,0.953235,0.962617,0.937763,0.948179,...,0.005469,0.989257,0.880995,0.931993,284225,1563,19442,143929,0.971586,0.968108
8,binary,dnn,fedavg_weighted,number_of_examples,cic_1,146314,0.878132,0.923220,0.758477,0.802489,...,0.002732,0.984430,0.519685,0.680259,109515,300,17531,18968,0.783511,0.726348
9,binary,dnn,fedavg_weighted,number_of_examples,cic_2,276449,0.994538,0.995625,0.992662,0.994110,...,0.000240,0.999581,0.985564,0.992523,174720,42,1468,100219,0.993823,0.994428
10,binary,dnn,fedavg_weighted,number_of_examples,cic_3,26396,0.577777,0.549005,0.778737,0.447196,...,0.000000,1.000000,0.557475,0.715870,1211,0,11145,14040,0.924970,0.996304
11,binary,dnn,fedavg_weighted,number_of_examples,pooled,449159,0.932126,0.950962,0.907145,0.923320,...,0.001197,0.997440,0.815487,0.897333,285446,342,30144,133227,0.948431,0.952099
12,binary,dnn,local_cic_1,None,cic_1,146314,0.877640,0.923104,0.757426,0.801458,...,0.002668,0.984725,0.517521,0.678472,109522,293,17610,18889,0.785932,0.726892
13,binary,dnn,local_cic_1,None,cic_2,276449,0.972002,0.978517,0.962146,0.969400,...,0.000566,0.998948,0.924858,0.960476,174663,99,7641,94046,0.993507,0.993186


{'study_identity': '4f71a14595de6e98fd10d2cfbb32b2ee9d6110fd0c88106f592111b41f32f981',
 'task': 'binary',
 'model': 'cnn',
 'trial': 6,
 'params': {'architecture': '4-8-4',
  'dropout': 0.3,
  'batch_size': 1024,
  'client_learning_rate': 0.0003,
  'kernel_size': 5},
 'rank': [0.7854105054080948, 0.6883147295293107, 0.8048760419764819],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'tuning_scenario': 'fedavg_weighted',
 'tuning_scenarios': ['local_cic_1',
  'local_cic_2',
  'local_cic_3',
  'centralized',
  'fedavg_weighted'],
 'objective': 'mean_cross_client_validation_f1'}

2026-11-03 19:44:02.118097: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 19:44:02.118147: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-03 19:44:02.118166: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-03 19:44:04.177512: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 19:44:04.177570: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register f

,task,model,scenario,client_weighting,test_client,samples,accuracy,macro_precision,macro_recall,macro_f1,...,far,precision,recall,f1,tn,fp,fn,tp,auc,pr_auc
0,binary,cnn,centralized,None,cic_1,146314,0.876984,0.912387,0.760556,0.802755,...,0.007094,0.961163,0.528206,0.681755,109036,779,17220,19279,0.785739,0.727834
1,binary,cnn,centralized,None,cic_2,276449,0.992552,0.992766,0.991212,0.991980,...,0.003719,0.993560,0.986144,0.989838,174112,650,1409,100278,0.993405,0.994299
2,binary,cnn,centralized,None,cic_3,26396,0.949803,0.614142,0.524463,0.533637,...,0.943848,0.956284,0.992773,0.974187,68,1143,182,25003,0.916444,0.992803
3,binary,cnn,centralized,None,pooled,449159,0.952393,0.960118,0.937929,0.947379,...,0.009000,0.982519,0.884857,0.931134,283216,2572,18811,144560,0.974373,0.969623
8,binary,cnn,fedavg_weighted,number_of_examples,cic_1,146314,0.877633,0.923100,0.757413,0.801444,...,0.002668,0.984724,0.517494,0.678448,109522,293,17611,18888,0.783843,0.726167
9,binary,cnn,fedavg_weighted,number_of_examples,cic_2,276449,0.993836,0.994612,0.992148,0.993357,...,0.001465,0.997453,0.985760,0.991572,174506,256,1448,100239,0.993884,0.994561
10,binary,cnn,fedavg_weighted,number_of_examples,cic_3,26396,0.799136,0.592968,0.894739,0.597962,...,0.000000,1.000000,0.789478,0.882356,1211,0,5302,19883,0.924717,0.996286
11,binary,cnn,fedavg_weighted,number_of_examples,pooled,449159,0.944541,0.958690,0.924482,0.937966,...,0.001921,0.996066,0.850885,0.917770,285239,549,24361,139010,0.945456,0.950950
12,binary,cnn,local_cic_1,None,cic_1,146314,0.877606,0.928580,0.755181,0.800077,...,0.000501,0.997059,0.510863,0.675580,109760,55,17853,18646,0.791877,0.732972
13,binary,cnn,local_cic_1,None,cic_2,276449,0.968566,0.976156,0.957384,0.965561,...,0.000315,0.999409,0.915083,0.955389,174707,55,8635,93052,0.993754,0.994183


In [2]:
import importlib, fedadam_search
importlib.reload(fedadam_search)
FEDADAM_CONFIG = PROJECT/"configs/fedadam_weighted.json"
fedadam_result = fedadam_search.run(FEDADAM_CONFIG, show_progress=True)
fedadam_trials = fedadam_search.tuning_table(FEDADAM_CONFIG)
display(fedadam_trials.style.format({
    "client_learning_rate": "{:.4g}", "server_learning_rate": "{:.4g}",
    "mean_validation_f1": "{:.2%}", "minimum_validation_f1": "{:.2%}",
    "mean_validation_recall": "{:.2%}", "mean_validation_far": "{:.4%}",
    "pooled_validation_f1": "{:.2%}"}))
print("Selected server LR:", fedadam_result["server_learning_rate"],
      "| Fixed client LR:", fedadam_result["params"]["client_learning_rate"])
display(fedadam_search.summary(fedadam_result).style.format({
    "samples": "{:,}", "accuracy": "{:.2%}", "precision": "{:.2%}", "recall": "{:.2%}",
    "f1": "{:.2%}", "far": "{:.4%}", "auc": "{:.2%}", "pr_auc": "{:.2%}"}))

Loaded completed, saved FedAdam results; all 20 validation trials and final model were reused. No model training was repeated.


Completed 20/20; best mean validation F1=0.85145


Completed 20/20; best mean validation F1=0.85145


Completed 20/20; best mean validation F1=0.85145


Completed 20/20; best mean validation F1=0.85145


Completed 20/20; best mean validation F1=0.85145


... (15 outputs left out)

Selected trial 20: fixed client LR=0.0003, server LR=0.1; retrain from scratch, then test once.


,rank,trial,client_learning_rate,server_learning_rate,mean_validation_f1,minimum_validation_f1,mean_validation_recall,mean_validation_far,pooled_validation_f1,selected
0,1,20,0.0003,0.1,85.14%,68.74%,76.56%,0.2365%,91.85%,True
1,2,18,0.0003,0.05,84.78%,69.10%,75.76%,0.1152%,91.77%,False
2,3,19,0.0003,0.07,84.09%,68.76%,74.72%,0.0945%,91.49%,False
3,4,17,0.0003,0.04,83.44%,68.89%,73.75%,0.1027%,91.17%,False
4,5,13,0.0003,0.007,82.67%,68.56%,73.20%,0.4568%,90.59%,False
5,6,12,0.0003,0.005,82.49%,68.10%,73.51%,0.8262%,90.24%,False
6,7,1,0.0003,0.01,81.85%,68.60%,71.83%,0.2859%,90.40%,False
7,8,11,0.0003,0.004,81.59%,66.88%,73.15%,1.4017%,89.43%,False
8,9,16,0.0003,0.03,81.52%,68.66%,70.97%,0.0615%,90.20%,False
9,10,14,0.0003,0.015,81.33%,68.65%,70.90%,0.1644%,90.14%,False


Selected server LR: 0.1 | Fixed client LR: 0.0003


,test_client,samples,accuracy,precision,recall,f1,far,auc,pr_auc
0,cic_1,"146,314",87.68%,97.37%,52.03%,67.82%,0.4662%,77.99%,72.27%
1,cic_2,"276,449",99.34%,99.59%,98.60%,99.09%,0.2357%,99.37%,99.45%
2,cic_3,"26,396",78.75%,100.00%,77.72%,87.47%,0.0000%,92.28%,99.62%
3,pooled,"449,159",94.33%,99.34%,84.98%,91.60%,0.3233%,94.56%,95.13%
